In [1]:
%pip install datasets

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
from datasets import load_dataset

import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer
from splade.splade.models.transformer_rep import Splade

df = load_dataset("microsoft/ms_marco", "v1.1")

/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
df

DatasetDict({
    validation: Dataset({
        features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
        num_rows: 10047
    })
    train: Dataset({
        features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
        num_rows: 82326
    })
    test: Dataset({
        features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
        num_rows: 9650
    })
})

In [4]:
model_type_or_dir = "naver/splade-cocondenser-ensembledistil"

model = Splade(model_type_or_dir, agg="max")
model.eval()

tokenizer = AutoTokenizer.from_pretrained(model_type_or_dir)
reverse_voc = {v: k for k, v in tokenizer.vocab.items()}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

/usr/local/lib/python3.10/dist-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your mod

In [5]:
from typing import List
from tqdm import tqdm
import time
import torch

mrr10 = 0

# Pre-compute batch size for tokenization
BATCH_SIZE = 32  # Adjust based on your GPU memory

for index, passage in tqdm(enumerate(df['train']['passages'][:100])):
    passage_text: List[str] = passage['passage_text']
    query: str = df['train']['query'][index]
    
    with torch.no_grad():
        start_time = time.time()
        query_tokens = tokenizer(query, return_tensors="pt").to(device)
        query_rep = model(d_kwargs=query_tokens)["d_rep"].squeeze()
        
        # Step 4: OPTIMIZED - Batch process all passages at once
        # Tokenize all passages in a single batch
        text_tokens = tokenizer(
            passage_text, 
            padding=True, 
            truncation=True,
            return_tensors="pt"
        ).to(device)
        
        # Get all document representations in one forward pass
        doc_rep = model(d_kwargs=text_tokens)["d_rep"]
        
        print("4: ", time.time() - start_time)
        
        # Step 5: OPTIMIZED - Avoid unnecessary tensor operations
        # Compute similarities
        similarities = query_rep @ doc_rep.T
        relevant_indices = torch.argsort(similarities, descending=True)
        
        # Find the index of the selected passage (avoid moving to GPU if already there)
        is_selected = passage['is_selected']
        result_index = is_selected.index(1) if isinstance(is_selected, list) else torch.argmax(torch.tensor(is_selected))
        
        print("5: ", time.time() - start_time)
        
        # Find rank
        rank = torch.where(relevant_indices == result_index)[0]
        
        if len(rank) > 0:
            mrr10 += 1 / (rank[0].item() + 1)

print(mrr10 / 100)


0it [00:00, ?it/s]

4:  2.33109188079834
5:  2.8051598072052


1it [00:03,  3.35s/it]


KeyboardInterrupt: 

In [21]:
import csv
import torch
from tqdm import tqdm
from collections import defaultdict
import pickle
import os
import time
from datetime import datetime

os.makedirs("backups", exist_ok=True)

reverse_index = defaultdict(list)
batch_size = 32

last_save_time = time.time()
save_interval = 30 * 60

with open("collection.tsv") as fd:
    rd = csv.reader(fd, delimiter="\t", quotechar='"')
    
    batch_docs = []
    batch_ids = []
    total_processed = 0
    
    for row in tqdm(rd):
        batch_ids.append(row[0])
        batch_docs.append(row[1])
        
        if len(batch_docs) == batch_size:
            passage_tokens = tokenizer(batch_docs, return_tensors="pt", truncation=True, 
                                      max_length=512, padding=True).to(device)
            
            with torch.no_grad():
                batch_reps = model(d_kwargs=passage_tokens)["d_rep"]
            
            for i, (doc_id, doc_rep) in enumerate(zip(batch_ids, batch_reps)):
                doc_rep = doc_rep.squeeze()
                mask = doc_rep > 0.01
                indices = torch.arange(doc_rep.size(0), device=device)[mask]
                weights = doc_rep[mask]
                
                sorted_indices = weights.argsort(descending=True)
                indices = indices[sorted_indices].cpu().numpy()
                weights = weights[sorted_indices].cpu().numpy()
                
                for idx, weight in zip(indices, weights):
                    reverse_index[reverse_voc[idx]].append((doc_id, float(weight)))
            
            total_processed += len(batch_docs)
            batch_docs = []
            batch_ids = []
            
            current_time = time.time()
            if current_time - last_save_time >= save_interval:
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                
                with open(f"backups/reverse_index_{timestamp}.pkl", "wb") as f:
                    pickle.dump(dict(reverse_index), f)
                
                with open(f"backups/progress_{timestamp}.txt", "w") as f:
                    f.write(f"Documents processed: {total_processed}\n")
                    f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
                
                print(f"\nBackup saved at {timestamp} - {total_processed} documents processed")
                last_save_time = current_time
    
    if batch_docs:
        passage_tokens = tokenizer(batch_docs, return_tensors="pt", truncation=True, 
                                  max_length=512, padding=True).to(device)
        
        with torch.no_grad():
            batch_reps = model(d_kwargs=passage_tokens)["d_rep"]
        
        for i, (doc_id, doc_rep) in enumerate(zip(batch_ids, batch_reps)):
            doc_rep = doc_rep.squeeze()
            mask = doc_rep > 0
            indices = torch.arange(doc_rep.size(0), device=device)[mask]
            weights = doc_rep[mask]
            
            sorted_indices = weights.argsort(descending=True)
            indices = indices[sorted_indices].cpu().numpy()
            weights = weights[sorted_indices].cpu().numpy()
            
            for idx, weight in zip(indices, weights):
                reverse_index[reverse_voc[idx]].append((doc_id, float(weight)))
        
        total_processed += len(batch_docs)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
with open(f"backups/reverse_index_final_{timestamp}.pkl", "wb") as f:
    pickle.dump(dict(reverse_index), f)

with open(f"backups/progress_final_{timestamp}.txt", "w") as f:
    f.write(f"Documents processed: {total_processed}\n")
    f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

print(f"\nProcessing complete. Total documents processed: {total_processed}")


1406720it [32:28,  1.73it/s] 


Backup saved at 20251103_130918 - 1406720 documents processed


2552608it [1:04:38,  1.56it/s] 


Backup saved at 20251103_133918 - 2552512 documents processed


3487296it [1:36:54,  1.10it/s] 


Backup saved at 20251103_140918 - 3487200 documents processed


4276064it [2:08:20,  1.27s/it] 


Backup saved at 20251103_143918 - 4275968 documents processed


4337759it [2:09:34, 557.95it/s]


KeyboardInterrupt: 

In [17]:
import sys

sys.getsizeof(reverse_index)

1310816

2597